# 05 – Spend Caps

Spend caps limit credits a user can consume per period. Useful for
cost governance in multi-tenant SaaS.

Cap actions:
- **deny** — reject when exceeded
- **warn** — allow but flag
- **notify** — allow and trigger event

> **Note:** This notebook uses `MemoryStore` because `PostgresStore`
> doesn't ship a `set_spend_cap` implementation. Caps are a
> store-specific feature.

In [ ]:
import uuid
from datetime import datetime, timedelta
from ducto.interface.memory import MemoryStore
from ducto.manager import CreditManager
from ducto.engine import PricingEngine
from ducto.metrics import UsageMetrics, ToolCall
from ducto.interface.models import (
    PricingConfigData, PricingConfigV2, PlanDefinition,
    CreditMetadata, SpendCap,
)

store = MemoryStore()
store.setup()
print("✔ MemoryStore ready.")

### Set a daily deny-cap of 5 000

In [ ]:
user = str(uuid.uuid4())
store.add_credits(user, 50_000, type="seed")
print(f"  Balance: {store.get_balance(user).balance}")

cap = SpendCap(user_id=user, cap_type="daily", limit=5_000, action="deny")
store.set_spend_cap(cap)
print(f"  Cap set: daily limit={cap.limit}, action={cap.action}")

### Deduct under cap (succeeds)

In [ ]:
res = store.reserve_credits(user, 3_000, operation_type="inference")
ded = store.deduct_credits(user, res.reservation_id, 3_000)
print(f"  Deducted 3k: balance={ded.balance_after}")
check = store.check_spend_cap(user)
print(f"  Cap check: capped={check.capped}, current={check.current_spend}")

### Exceed cap (denied)

In [ ]:
res2 = store.reserve_credits(user, 3_000, operation_type="inference")
ded2 = store.deduct_credits(user, res2.reservation_id, 3_000)
if ded2.error:
    print(f"  Denied: {ded2.error}")
else:
    print(f"  Allowed: {ded2.balance_after}")
print("  (daily cap = 5k, already spent 3k)")

### Cap with warn action

In [ ]:
user2 = str(uuid.uuid4())
store.add_credits(user2, 50_000, type="seed")
store.set_spend_cap(SpendCap(user_id=user2, cap_type="daily", limit=500, action="warn"))

res3 = store.reserve_credits(user2, 1_000, operation_type="inference")
ded3 = store.deduct_credits(user2, res3.reservation_id, 1_000)
check2 = store.check_spend_cap(user2)
print(f"  Current: {check2.current_spend}  Cap: {check2.cap_limit}  Action: {check2.action}")
print("  (warn — deduction still went through)")